# H17: Direction vs. Conditioning -- Which Split Predicts Architecture Benefit?

## Theoretical Motivation

The Muon optimizer applies Newton-Schulz orthogonalization to each gradient matrix,
replacing $G = U\Sigma V^T$ with $UV^T$. This does two things simultaneously:

1. **Direction change**: The update direction rotates from the raw gradient toward the
   nearest orthogonal matrix (polar factor).
2. **Conditioning change**: All singular values are equalized to 1, removing the spectral
   spread that causes SGD to make poorly-conditioned steps.

A key question is: **which of these two effects drives Muon's architecture-dependent benefit?**

## Hypothesis

> **H17 (Direction-vs-Conditioning Split):** Architectures with higher gradient anisotropy
> ($\sigma_1/\sigma_{\min}$ of the gradient, averaged across layers) benefit more from Muon,
> because there are more singular values to equalize. Specifically:
>
> - **T1:** Pearson correlation between anisotropy and Muon advantage $r > 0.5$
> - **T2:** The architecture with the highest gradient anisotropy also shows the highest
>   Muon advantage

## Why "Direction vs. Conditioning" Matters

Prior experiments in this project (e.g., hybrid networks, H21) found that Muon layers
sometimes have **worse** conditioning ($\kappa \sim 0.8\times$ SGD) but **3x better**
directional quality. This suggests:

- If anisotropy predicts benefit, the mechanism is **spectral equalization** (conditioning path)
- If anisotropy does NOT predict benefit, the mechanism is primarily **directional** --
  Muon finds better descent directions independent of the spectral shape

This experiment tests the conditioning path by checking whether initial gradient
anisotropy (a pure spectral property) correlates with the eventual training advantage.

## Experimental Design

| Architecture   | Structure                  | Activation | Expected Anisotropy          |
|----------------|----------------------------|------------|------------------------------|
| deep_linear    | 4 x (32, 32), I + 0.1N    | none       | low (near-identity init)     |
| relu_net       | 4 x (32, 32), I + 0.1N    | ReLU       | moderate (dead neurons)      |
| tanh_net       | 4 x (32, 32), I + 0.1N    | tanh       | moderate (saturation)        |
| bottleneck     | 4 x (32, 32), SVD-crushed | ReLU       | high (bottom SVs suppressed) |

**Key difference from H17_ANISOTROPY_PREDICTS_BENEFIT:** This experiment uses **mean
anisotropy across all layers** (not just layer 0) and has a distinctive bottleneck
architecture where half the singular values are explicitly crushed to 0.01x at init.

## Environment Setup

Pure NumPy implementation -- no autograd framework. All gradients are computed by
explicit backpropagation, making the computation fully transparent and auditable.

In [ ]:
import numpy as np
import os

In [ ]:
SCRIPT_DIR = os.path.dirname(os.path.abspath('.'))


## Experimental Configuration

Key hyperparameters:

- **DIM = 32**: All weight matrices are 32x32 (square), keeping the geometry uniform
  across architectures so that anisotropy differences come from activations and init, not shape.
- **NUM_LAYERS = 4**: Deep enough for depth effects (gradient spectrum distortion) but
  shallow enough to avoid vanishing gradients with the near-identity init.
- **NUM_STEPS = 500**: Longer than H17_ANISOTROPY (300 steps) to ensure convergence
  differences are meaningful and not just transient.
- **MOMENTUM = 0.9**: Identical for SGD and Muon -- the only difference is Newton-Schulz.
- **NS_ITERS = 5**: Newton-Schulz iterations for polar decomposition.
- **NUM_SEEDS = 5**: Five independent random seeds for statistical averaging.
- **BATCH_SIZE = 64**: Fixed batch to isolate optimizer effects from stochastic noise.

In [ ]:
DIM = 32
NUM_LAYERS = 4
NUM_STEPS = 500
MOMENTUM = 0.9
NS_ITERS = 5
NUM_SEEDS = 5
BATCH_SIZE = 64

print("\n--- Experimental Configuration ---")
print(f"  DIM            = {DIM}    (square weight matrices)")
print(f"  NUM_LAYERS     = {NUM_LAYERS}     (network depth)")
print(f"  NUM_STEPS      = {NUM_STEPS}   (training iterations per run)")
print(f"  BATCH_SIZE     = {BATCH_SIZE}    (fixed batch, no stochastic noise)")
print(f"  MOMENTUM       = {MOMENTUM}  (heavy-ball, same for SGD and Muon)")
print(f"  NUM_SEEDS      = {NUM_SEEDS}     (independent random initializations)")
print(f"  NS_ITERS       = {NS_ITERS}     (Newton-Schulz iterations for Muon)")
print(f"\n  Total LR sweep budget: 4 archs x (8+8) LR candidates x 3 seeds = {4*16*3} runs")
print(f"  Total full-eval runs:  4 archs x 2 optimizers x 5 seeds = {4*2*5} runs")

### Learning Rate Search Space

Independent LR grids for Muon and SGD. Muon typically operates at lower LR because
the Newton-Schulz step normalizes the gradient magnitude (all SVs become 1), giving
updates with Frobenius norm $\approx \sqrt{\min(m,n)}$ regardless of gradient scale.

In [ ]:
LR_MUON = [0.05, 0.03, 0.02, 0.015, 0.01, 0.007, 0.005, 0.003]
LR_SGD = [0.2, 0.1, 0.05, 0.03, 0.02, 0.01, 0.005, 0.003]

## Newton-Schulz Orthogonalization (Muon's Core)

The Newton-Schulz iteration computes the polar factor $UV^T$ of the gradient
$G = U\Sigma V^T$, setting all singular values to 1. The iteration:

$$X_{k+1} = \frac{3}{2}X_k - \frac{1}{2}X_k(X_k^TX_k)$$

This is the **sole difference** between SGD and Muon in this experiment. Every other
aspect (momentum, LR application, weight update) is identical.

In [ ]:
def newton_schulz(M, n_iters=NS_ITERS):
    norm = np.linalg.norm(M, ord='fro')
    if norm < 1e-15:
        return M
    X = M / norm
    for _ in range(n_iters):
        A = X.T @ X
        X = 1.5 * X - 0.5 * X @ A
    return X

## Weight Initialization and Architecture Definitions

The four architectures share the same 32x32 dimensionality but differ in initialization
and activation:

- **deep_linear / relu_net / tanh_net**: $W = I + 0.1 \cdot \mathcal{N}(0,1)$ -- a small
  perturbation around identity. This near-identity init keeps the gradient spectrum
  well-conditioned initially, so anisotropy differences come from the activation function.
- **bottleneck**: SVD-based construction where the bottom half of singular values are
  crushed to $0.01\times$, plus $0.5I$. This **explicitly manufactures** high anisotropy
  in the weight spectrum, which propagates to gradient anisotropy.

The bottleneck design tests whether artificially high anisotropy translates to
proportionally higher Muon benefit, as the spectral equalization theory predicts.

In [ ]:
def init_weights(arch, seed):
    rng = np.random.RandomState(seed)
    if arch == 'deep_linear':
        return [np.eye(DIM) + rng.randn(DIM, DIM) * 0.1 for _ in range(NUM_LAYERS)]
    elif arch == 'relu_net':
        return [np.eye(DIM) + rng.randn(DIM, DIM) * 0.1 for _ in range(NUM_LAYERS)]
    elif arch == 'tanh_net':
        return [np.eye(DIM) + rng.randn(DIM, DIM) * 0.1 for _ in range(NUM_LAYERS)]
    elif arch == 'bottleneck':
        # Bottleneck: layers 1,3 are wide->narrow, 2,4 are narrow->wide
        # Using rectangular-ish SVD structure
        weights = []
        for l in range(NUM_LAYERS):
            W = rng.randn(DIM, DIM) * 0.1
            # Make half the SVs very small (simulated bottleneck)
            U, s, Vt = np.linalg.svd(W, full_matrices=False)
            s[DIM//2:] *= 0.01  # suppress bottom half
            W = U @ np.diag(s) @ Vt
            W += np.eye(DIM) * 0.5
            weights.append(W)
        return weights

In [ ]:
def apply_act(x, arch, layer_idx, num_layers):
    if arch == 'deep_linear' or layer_idx == num_layers - 1:
        return x
    elif arch == 'relu_net':
        return np.maximum(0, x)
    elif arch == 'tanh_net':
        return np.tanh(x)
    elif arch == 'bottleneck':
        return np.maximum(0, x)

In [ ]:
def act_deriv(pre, arch, layer_idx, num_layers):
    if arch == 'deep_linear' or layer_idx == num_layers - 1:
        return np.ones_like(pre)
    elif arch == 'relu_net' or arch == 'bottleneck':
        return (pre > 0).astype(float)
    elif arch == 'tanh_net':
        return 1 - np.tanh(pre)**2

## Forward Pass, Loss, and Gradient Computation

Standard feedforward network: $\hat{y} = \phi(W_L \cdot \phi(W_{L-1} \cdots \phi(W_1 x)))$
with MSE loss $\mathcal{L} = \frac{1}{2N}\sum_n \|\hat{y}_n - y_n\|^2$. Gradients are
computed by explicit backpropagation (no autograd).

In [ ]:
def forward(weights, X, arch):
    out = X.copy()
    for idx, W in enumerate(weights):
        out = W @ out
        out = apply_act(out, arch, idx, len(weights))
    return out

In [ ]:
def compute_loss(weights, X, Y, arch):
    pred = forward(weights, X, arch)
    return 0.5 * np.mean(np.sum((pred - Y)**2, axis=0))

In [ ]:
def compute_gradients(weights, X, Y, arch):
    L = len(weights)
    N = X.shape[1]
    acts_post = [X.copy()]
    pre_acts = []
    out = X.copy()
    for idx, W in enumerate(weights):
        pre = W @ out
        pre_acts.append(pre)
        out = apply_act(pre, arch, idx, L)
        acts_post.append(out)
    delta = (acts_post[-1] - Y) / N
    grads = [None] * L
    for l in range(L - 1, -1, -1):
        grads[l] = delta @ acts_post[l].T
        if l > 0:
            delta = weights[l].T @ delta
            delta = delta * act_deriv(pre_acts[l-1], arch, l-1, L)
    return grads

## Gradient Anisotropy Measurement

Unlike H17_ANISOTROPY_PREDICTS_BENEFIT (which uses only layer 0), this experiment
computes the **mean anisotropy across all layers**:

$$\text{anisotropy} = \frac{1}{L} \sum_{l=1}^{L} \frac{\sigma_1(\nabla_{W_l}\mathcal{L})}{\sigma_{\min}(\nabla_{W_l}\mathcal{L})}$$

This captures the full network's gradient geometry, not just the input layer.
The rationale: Muon orthogonalizes every layer's gradient, so the total benefit should
depend on anisotropy at all layers.

In [ ]:
def gradient_anisotropy(weights, X, Y, arch):
    """Mean sigma_1 / sigma_min of gradient across layers."""
    grads = compute_gradients(weights, X, Y, arch)
    anisotropies = []
    for G in grads:
        s = np.linalg.svd(G, compute_uv=False)
        if s[-1] > 1e-12:
            anisotropies.append(s[0] / s[-1])
        else:
            anisotropies.append(s[0] / 1e-12)
    return np.mean(anisotropies)

## Training Engine and Data Generation

The training loop is identical for SGD and Muon except for the Newton-Schulz step.
Both use heavy-ball momentum with $\beta = 0.9$.

Data is random Gaussian ($\sigma = 0.3$) -- the task is pure regression on noise,
so architecture geometry is the only variable driving the anisotropy-advantage relationship.

In [ ]:
def train(weights_init, X, Y, lr, optimizer, arch):
    weights = [W.copy() for W in weights_init]
    mom = [np.zeros_like(W) for W in weights]
    for step in range(NUM_STEPS):
        loss = compute_loss(weights, X, Y, arch)
        if not np.isfinite(loss) or loss > 1e10:
            return float('inf')
        grads = compute_gradients(weights, X, Y, arch)
        for i in range(len(weights)):
            if optimizer == 'muon':
                mom[i] = MOMENTUM * mom[i] + newton_schulz(grads[i])
            else:
                mom[i] = MOMENTUM * mom[i] + grads[i]
            weights[i] = weights[i] - lr * mom[i]
    return compute_loss(weights, X, Y, arch)

In [ ]:
def make_data(seed):
    rng = np.random.RandomState(seed)
    X = rng.randn(DIM, BATCH_SIZE) * 0.3
    Y = rng.randn(DIM, BATCH_SIZE) * 0.3
    return X, Y

### Diagnostic: Inspect Per-Layer Gradient Anisotropy at Initialization

Before the main experiment, let us inspect the gradient anisotropy at each layer for
each architecture. This reveals whether the bottleneck architecture truly produces
higher gradient anisotropy (as designed) and whether nonlinear activations affect
the spectrum differently across layers.

In [ ]:
print("=" * 80)
print("DIAGNOSTIC: Per-Layer Gradient Anisotropy at Initialization (seed=42)")
print("=" * 80)
diag_archs = ['deep_linear', 'relu_net', 'tanh_net', 'bottleneck']
for arch in diag_archs:
    X, Y = make_data(42)
    w = init_weights(arch, 42 + 5000)
    grads = compute_gradients(w, X, Y, arch)
    print(f"\n  {arch:>15}:")
    layer_anisos = []
    for l, G in enumerate(grads):
        sv = np.linalg.svd(G, compute_uv=False)
        aniso = sv[0] / max(sv[-1], 1e-12)
        layer_anisos.append(aniso)
        print(f"    Layer {l}: sigma_1={sv[0]:.6f}  sigma_min={sv[-1]:.6f}  ratio={aniso:.2f}")
    print(f"    Mean anisotropy across layers: {np.mean(layer_anisos):.2f}")
print("\n" + "=" * 80)
print("Expect: bottleneck >> relu_net >= tanh_net >= deep_linear")
print("=" * 80)

---
## Main Experiment: Anisotropy vs. Muon Advantage Across Architectures

The main function runs the full pipeline:

1. For each architecture, measure mean gradient anisotropy at init (averaged over 5 seeds)
2. LR sweep for both SGD and Muon (8 candidates each, 3 seeds for speed)
3. Full training at best LR with all 5 seeds
4. Compute Muon advantage = $\text{loss}_{\text{SGD}} / \text{loss}_{\text{Muon}}$
5. Compute Pearson correlation between anisotropy and advantage
6. Run hypothesis tests (T1: $r > 0.5$, T2: rank-1 match)

In [ ]:
def main():
    seeds = [42 + i * 137 for i in range(NUM_SEEDS)]
    archs = ['deep_linear', 'relu_net', 'tanh_net', 'bottleneck']

    print("=" * 100)
    print("H17: DIRECTION vs CONDITIONING SPLIT -- Architecture Benefit Predictor")
    print("=" * 100)
    print(f"Architectures: {archs}")
    print(f"Network: {NUM_LAYERS}-layer, {DIM}x{DIM}, {NUM_STEPS} steps, {NUM_SEEDS} seeds")
    print()

    results = {}
    for arch in archs:
        print(f"\n--- {arch.upper()} ---")

        # Measure gradient anisotropy at init
        anisotropies = []
        for s in seeds:
            X, Y = make_data(s)
            w = init_weights(arch, s + 5000)
            aniso = gradient_anisotropy(w, X, Y, arch)
            anisotropies.append(aniso)
        mean_aniso = np.mean(anisotropies)
        print(f"  Gradient anisotropy: {mean_aniso:.2f}")

        # LR sweep
        best = {}
        for opt, candidates in [('muon', LR_MUON), ('sgd', LR_SGD)]:
            best_lr, best_loss = candidates[-1], float('inf')
            for lr in candidates:
                losses = []
                for s in seeds[:3]:
                    X, Y = make_data(s)
                    w = init_weights(arch, s + 5000)
                    fl = train(w, X, Y, lr, opt, arch)
                    losses.append(fl)
                ml = np.mean([l for l in losses if np.isfinite(l)]) if any(np.isfinite(l) for l in losses) else float('inf')
                if ml < best_loss:
                    best_loss = ml
                    best_lr = lr
            best[opt] = best_lr

        # Full training
        for opt in ['muon', 'sgd']:
            losses = []
            for s in seeds:
                X, Y = make_data(s)
                w = init_weights(arch, s + 5000)
                fl = train(w, X, Y, best[opt], opt, arch)
                losses.append(fl)
            finite = [l for l in losses if np.isfinite(l)]
            results[(arch, opt)] = np.mean(finite) if finite else float('inf')

        advantage = results[(arch, 'sgd')] / max(results[(arch, 'muon')], 1e-30)
        results[(arch, 'anisotropy')] = mean_aniso
        results[(arch, 'advantage')] = advantage
        print(f"  Muon advantage: {advantage:.1f}x")

    # Correlation analysis
    print(f"\n\n{'=' * 100}")
    print("CORRELATION: Gradient Anisotropy vs Muon Advantage")
    print(f"{'=' * 100}")

    print(f"\n  {'Architecture':>15}  {'Anisotropy':>12}  {'Advantage':>12}")
    print("  " + "-" * 42)

    anisos = []
    advs = []
    for arch in archs:
        a = results[(arch, 'anisotropy')]
        adv = results[(arch, 'advantage')]
        anisos.append(a)
        advs.append(adv)
        print(f"  {arch:>15}  {a:>12.2f}  {adv:>12.1f}x")

    corr = np.corrcoef(anisos, advs)[0, 1]
    print(f"\n  Correlation r = {corr:.3f}")

    # Hypothesis tests
    print(f"\n{'=' * 100}")
    print("HYPOTHESIS TESTS")
    print(f"{'=' * 100}")

    t1 = corr > 0.5
    print(f"\n  T1: Positive correlation r > 0.5?")
    print(f"      r = {corr:.3f}")
    print(f"      --> {'PASS' if t1 else 'FAIL'}")

    max_aniso_arch = archs[np.argmax(anisos)]
    max_adv_arch = archs[np.argmax(advs)]
    t2 = max_aniso_arch == max_adv_arch
    print(f"\n  T2: Highest anisotropy = highest advantage?")
    print(f"      Max anisotropy: {max_aniso_arch}")
    print(f"      Max advantage:  {max_adv_arch}")
    print(f"      --> {'PASS' if t2 else 'FAIL'}")

    print(f"\n{'=' * 100}")
    print("EXPERIMENT COMPLETE")
    print(f"{'=' * 100}")

In [ ]:
if __name__ == '__main__':
    main()

### Interpretation of Results

**Reading the output above:**

- The correlation table shows each architecture's mean gradient anisotropy (across all
  4 layers) and the Muon advantage ratio ($\text{loss}_{\text{SGD}} / \text{loss}_{\text{Muon}}$).
- **T1 ($r > 0.5$)** tests whether the Pearson correlation confirms a positive linear
  relationship between anisotropy and benefit.
- **T2** tests whether the architecture with the most anisotropic gradients also gets
  the largest Muon advantage -- a necessary condition for the spectral equalization theory.

**Key question this experiment answers:**
Does knowing the gradient spectrum geometry tell you *which architectures* benefit from Muon?
If yes, the benefit is about conditioning (spectral equalization). If no, the benefit is
about direction quality (the polar factor finding better descent directions regardless
of the spectral shape).

## Conclusions

### What This Experiment Tests

**H17 (Direction vs. Conditioning)** asks whether gradient anisotropy -- a purely
spectral property measurable before training -- predicts which architectures benefit
most from Muon. This directly tests the "spectral equalization" interpretation of
Muon's mechanism.

### Relationship to H17_ANISOTROPY_PREDICTS_BENEFIT

This experiment differs from the sibling H17 experiment in three ways:
1. **Mean anisotropy across all layers** (not just layer 0)
2. **Different bottleneck design** -- SVD-crushed singular values rather than rectangular matrices
3. **Fewer architectures** (4 instead of 5) -- no "wide" architecture

The two experiments together provide convergent evidence: if both confirm the anisotropy-
advantage correlation with different measurement methods, the result is robust.

### Implications for the "Direction vs. Conditioning" Question

- **If T1 passes (r > 0.5):** The spectral equalization (conditioning) path is a significant
  driver of Muon's benefit. Architectures with more ill-conditioned gradients benefit
  proportionally from having their singular values equalized.
- **If T1 fails (r <= 0.5):** The directional quality path dominates. Muon helps not because
  it fixes spectral imbalance, but because the polar factor $UV^T$ is simply a better
  descent direction than the raw gradient, for reasons beyond spectral conditioning.

### Caveats

1. **Only 4 data points** -- Pearson correlation with 4 points is inherently noisy.
   A single outlier architecture can flip the sign of $r$.
2. **Anisotropy is measured at init only** -- it may change rapidly during training,
   making init anisotropy a poor predictor of long-run benefit.
3. **The bottleneck architecture is designed to have high anisotropy** -- this is not
   a natural architecture but an engineered stress test. If it dominates the correlation,
   the result may not generalize to practical architectures.

### Connection to the Broader Framework

This experiment addresses a fundamental question in the Muon-as-RG-Gauge-Fixing theory:
is the gauge being "fixed" a spectral gauge (singular value imbalance) or a directional
gauge (gradient misalignment with the loss landscape geometry)? The answer determines
which theoretical interpretation is primary.